Attempting to apply Gauss Seidel to $x_0=\sin(\pi x)\sin(\pi y)+\frac{1}{10}\sin(10\pi x)\sin(10\pi y)$  
Then attempting to create two levels of finite element spaces such that one is a coarser version of the other and make a prolongation P matrix which projects from one space to the other.  
Overall goal is to:  
1. apply a custom number of GS smoothing steps to our initial iterate $x_0$,  
2. visualize the current iterate after such smoothing,   
3. then project onto the coarse level,   
4. display the iterate in its coarse form,   
5. direct solve,   
6. then project back to the fine level and   
7. view the solution.

In [13]:
# Import any packages needed
from ngsolve import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
import numpy as np

In [14]:
# Create two levels of finite element spaces

coarse_mesh = Mesh(unit_square.GenerateMesh(maxh=1/16))
fesc = H1(coarse_mesh, order=1, dirichlet="left|right")
fine_mesh = Mesh(coarse_mesh.ngmesh.Copy()) # they are the same size right now
fesf = H1(fine_mesh, order=1, dirichlet="left|right")

Draw(coarse_mesh)


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [15]:
# Changing the "fine" space to have the higher dimension by refining the mesh
# this approach allows for the creating of the prolongation matrix, 
# if we create a finite element space from a finer mesh, it doesn't have the necessary levels
fesf.mesh.Refine()

Draw(fesf.mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [16]:
# Confirming that the coarse mesh has not been modified
Draw(fesc.mesh)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [17]:
# Generating and storing the projection matrices for moving things between the coarse and fine levels
P = fesf.Prolongation().CreateMatrix(fesf.mesh.levels-1)
PT = P.CreateTranspose()
PT.shape
#cprint(PT)

(339, 1289)

In [18]:
# Setup the PDE for solving, just focusing on the Poisson problem for this stage of the project
u, v = fesf.TnT()
a = BilinearForm(grad(u)*grad(v)*dx).Assemble()
f = LinearForm(v*dx).Assemble()
#print(a.mat)
#print(f.vec)


In [19]:
# Creating Gauss Seidel smoother
a_fine = a.GetMatrixLevel()
#a_fine.shape
m = a_fine.CreateSmoother(fesf.FreeDofs(), GS=True)
# made it smaller and lost the part that made it invertible

In [20]:
# From GS Smoother in 2.1.2 of iTutorials
# haven't tested this cell yet, needs editing, but wanted to save it
# gfu.vec[:] = 0
# res = f.vec.CreateVector()              # residual
# projres = f.vec.CreateVector()          # residual projected to freedofs
# proj = Projector(fes.FreeDofs(), True)

# for i in range(500):
#     preJpoint.Smooth(gfu.vec, f.vec)    # one step of point Gauss-Seidel
#     res.data = f.vec - a.mat*gfu.vec
#     projres.data = proj * res
#     print ("it#", i, ", res =", Norm(projres))
# Draw (gfu)

In [21]:
# In the fine level, setting up the initial iterate
x0 = CoefficientFunction(sin(pi*x)*sin(pi*y)+(1/10)*sin(10*pi*x)*sin(10*pi*y))
xi = GridFunction(fesf)
xi.Set(x0)
Draw(xi,fine_mesh,deformation=True)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [22]:
# s = CoefficientFunction(sin(pi*x)*sin(pi*y)+(1/10)*sin(10*pi*x)*sin(10*pi*y))
# st = GridFunction(fesf)
# st.Set(10, BND)
# st.Set(s)
# Draw(st, fine_mesh, deformation=True)
# Look into Dirichlet boundary conditions and functions over writing

In [23]:
# Applying the GS smoothing to the vector function we made
yi = GridFunction(fesf)
# Draw(xi, fine_mesh, Deformation=True)
# zi = GridFunction(fesf)
smootha = m @ a_fine
# we need to confirm that this is accomplishing what we intend
for i in range(10):
    smootha.Mult(xi.vec, yi.vec.data)
    xi.vec.data -= yi.vec
    if i%2 == 0:
        Draw(xi,fine_mesh,deformation=True) 

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [36]:
# Once we have successfully smoothed the fine level, we go to the coarse level for solving
a_coarse = PT @ a_fine @ P
a_coarse.shape
constance = GridFunction(fesc)
constance.Set(1)
outvect = GridFunction(fesc)
a_coarse.Mult(constance.vec, outvect.vec.data)
print(outvect.vec)

#print(a_coarse)
# check a_coarse against the assembled a from the coarse fes


 -1.11022e-16
 -4.16334e-16
       0
 -2.77556e-16
       0
 2.22045e-16
       0
 4.44089e-16
 3.33067e-16
 4.44089e-16
 8.88178e-16
 6.66134e-16
 1.11022e-16
 2.22045e-16
 1.11022e-16
 -1.11022e-16
 7.77156e-16
 -3.33067e-16
 4.44089e-16
 8.32667e-16
 -3.33067e-16
 3.88578e-16
 4.996e-16
 4.44089e-16
 2.22045e-16
 4.996e-16
 -2.22045e-16
 3.33067e-16
 -1.33227e-15
 -1.11022e-16
 6.66134e-16
 -2.22045e-16
 -6.66134e-16
 -2.22045e-16
 -1.15699e-15
       0
 -3.33067e-16
 3.33067e-16
 -1.22125e-15
 -6.66134e-16
 4.44089e-16
 -2.22045e-16
 6.66134e-16
 4.44089e-16
 3.33067e-16
 7.77156e-16
 -1.22125e-15
 -6.66134e-16
 1.11022e-16
 -9.4369e-16
 -5.55112e-16
 1.66533e-16
 -6.10623e-16
 -2.77556e-16
 1.11022e-16
 -1.11022e-16
 6.66134e-16
 1.11022e-16
 -2.22045e-16
 -1.11022e-16
 -3.33067e-16
 3.88578e-16
 6.66134e-16
 -4.44089e-16
 -6.66134e-16
 3.33067e-16
 -7.77156e-16
 1.22125e-15
 7.77156e-16
 3.33067e-16
 -7.77156e-16
 -4.44089e-16
 -1.11022e-16
 -8.88178e-16
 1.11022e-15
 -1.11022e-1

In [ ]:
# Making the coarse inverse for solving
acinv = a_coarse.Inverse(fesc.FreeDofs())

len(f.vec)
fc = GridFunction(fesc)
fc.vec.data = PT * f.vec
len(fc.vec)
yc = GridFunction(fesc)
yc.vec.data = PT * xi.vec # this supposes that the result of the GS smoothing is a vector xi of the "fine" length
xc = GridFunction(fesc)
Draw(yc, coarse_mesh, deformation=True)


In [ ]:
xc.vec.data = acinv * yc.vec
Draw(xc, coarse_mesh, deformation=True)

In [ ]:
# Take the solution back to the fine level
ufine = GridFunction(fesf)
ufine.vec.data = P * xc.vec
Draw(ufine, fine_mesh, Deformation=True)